# 08 — Advanced RUL Models: LSTM / GRU / Temporal CNN / Transformer (FD001)

**Phase 7 — Advanced RUL Models**

Goals:
- Build fixed-length sliding-window sequences from the engineered features
- Train LSTM, GRU, Temporal CNN, and Transformer regressors
- Compare all four against each other AND against the tuned XGBoost baseline from notebook 07

In [1]:
import sys
sys.path.append('..')

import json
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.config.config import PROCESSED_DATA_DIR, SEQUENCE_LENGTH, RUL_MODELS_DIR
from src.data.transformation import create_sequences, engine_train_val_split
from src.models.sequence_models import (
    ARCHITECTURES, RULSequenceDataset, build_model, train_sequence_model,
    predict_sequence_model, save_sequence_model,
)
from src.models.evaluate import compute_metrics, compare_models
from src.models.predict import load_model as load_baseline_model

VARIANT = 'FD001'
train_fe = pd.read_parquet(PROCESSED_DATA_DIR / f'train_{VARIANT}_features.parquet')
val_fe = pd.read_parquet(PROCESSED_DATA_DIR / f'val_{VARIANT}_features.parquet')
with open(PROCESSED_DATA_DIR / f'selected_features_{VARIANT}.json') as f:
    selected_features = json.load(f)['selected_features']

print(f'Sequence length: {SEQUENCE_LENGTH} cycles, {len(selected_features)} features/cycle')

Sequence length: 30 cycles, 232 features/cycle


## Build sliding-window sequences
Each sample is `SEQUENCE_LENGTH` consecutive cycles from ONE engine; the label is the RUL at the last cycle of the window. Engines shorter than the window are left-padded (see `create_sequences` docstring).

In [2]:
X_train, y_train, train_ids = create_sequences(
    train_fe, selected_features, 'RUL_clipped', SEQUENCE_LENGTH,
)
X_val, y_val, val_ids = create_sequences(
    val_fe, selected_features, 'RUL_clipped', SEQUENCE_LENGTH,
)
print('Train sequences:', X_train.shape)
print('Val sequences  :', X_val.shape)

[2026-10-09 07:16:16] INFO     | src.data.transformation | Created 14241 sequences of length 30 with 232 features
[2026-10-09 07:16:16] INFO     | src.data.transformation | Created 3490 sequences of length 30 with 232 features
Train sequences: (14241, 30, 232)
Val sequences  : (3490, 30, 232)


In [3]:
train_loader = DataLoader(RULSequenceDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(RULSequenceDataset(X_val, y_val), batch_size=64, shuffle=False)

## Train all four architectures
Each model trains with early stopping on validation loss (see `src/models/sequence_models.py::train_sequence_model`). Increase `epochs` for a real run — this default is kept modest so the notebook finishes quickly; watch the training loss curves below to judge whether more epochs would help.

In [ ]:
histories = {}
fitted_models = {}
n_features = len(selected_features)

for arch in ARCHITECTURES:
    print(f'--- Training {arch} ---')
    model = build_model(arch, n_features=n_features)
    history = train_sequence_model(
        model, train_loader, val_loader, epochs=30, patience=5,
    )
    histories[arch] = history
    fitted_models[arch] = model
    save_sequence_model(
        model, arch, {'n_features': n_features},
        RUL_MODELS_DIR / f'{arch}_{VARIANT}.pt',
    )

--- Training lstm ---
[2026-10-09 07:16:20] INFO     | src.models.sequence_models | Epoch 1/30 — train_loss=4812.5098, val_loss=1864.6535
[2026-10-09 07:16:23] INFO     | src.models.sequence_models | Epoch 2/30 — train_loss=1883.3603, val_loss=1743.6691
[2026-10-09 07:16:26] INFO     | src.models.sequence_models | Epoch 3/30 — train_loss=1884.4353, val_loss=1744.5440
[2026-10-09 07:16:29] INFO     | src.models.sequence_models | Epoch 4/30 — train_loss=1903.1708, val_loss=1743.3844
[2026-10-09 07:16:32] INFO     | src.models.sequence_models | Epoch 5/30 — train_loss=1879.4654, val_loss=1745.6308
[2026-10-09 07:16:35] INFO     | src.models.sequence_models | Epoch 6/30 — train_loss=1877.3603, val_loss=1743.7328
[2026-10-09 07:16:39] INFO     | src.models.sequence_models | Epoch 7/30 — train_loss=1883.3275, val_loss=1746.0849
[2026-10-09 07:16:42] INFO     | src.models.sequence_models | Epoch 8/30 — train_loss=1884.0292, val_loss=1744.7833
[2026-10-09 07:16:45] INFO     | src.models.sequen

## Training curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (arch, history) in zip(axes.flatten(), histories.items()):
    ax.plot(history.train_loss, label='train')
    ax.plot(history.val_loss, label='val')
    ax.axvline(history.best_epoch - 1, color='red', linestyle='--', alpha=0.5, label='best epoch')
    ax.set_title(arch); ax.set_xlabel('epoch'); ax.set_ylabel('MSE loss'); ax.legend()
plt.tight_layout(); plt.show()

## Evaluate all sequence models on validation

In [ ]:
dl_results = {}
dl_predictions = {}
for arch, model in fitted_models.items():
    preds = predict_sequence_model(model, X_val)
    dl_results[arch] = compute_metrics(y_val, preds)
    dl_predictions[arch] = preds

compare_models(dl_results)

## Compare against the tuned XGBoost baseline (notebook 07)
**Note:** XGBoost is trained/evaluated on individual engineered rows, while the sequence models are trained/evaluated on sliding-window sequences — the validation engines are the same, but the exact sample counts differ (a sequence "uses up" `SEQUENCE_LENGTH - 1` extra rows of history per engine). Treat this as an approximate, not perfectly row-matched, comparison.

In [ ]:
xgb_model, xgb_features = load_baseline_model('xgboost', variant=VARIANT)
xgb_val_row_level = val_fe[xgb_features]
xgb_preds = xgb_model.predict(xgb_val_row_level)
xgb_metrics = compute_metrics(val_fe['RUL_clipped'], xgb_preds)

all_results = dict(dl_results)
all_results['xgboost_tuned'] = xgb_metrics
final_comparison = compare_models(all_results)
final_comparison

In [ ]:
final_comparison['RMSE'].plot(
    kind='bar', figsize=(8, 4), color='slateblue',
    title='RMSE across all Phase 6 + Phase 7 models — FD001',
)
plt.ylabel('RMSE (cycles)'); plt.xticks(rotation=30)
plt.tight_layout(); plt.show()

final_comparison.to_csv('../reports/metrics/full_model_comparison_FD001.csv')
print('Saved combined comparison to reports/metrics/full_model_comparison_FD001.csv')

## Actual vs. predicted — best overall model

In [ ]:
best_overall = final_comparison.index[0]
print('Best model overall by RMSE:', best_overall)

if best_overall in dl_predictions:
    preds_to_plot, y_to_plot = dl_predictions[best_overall], y_val
else:
    preds_to_plot, y_to_plot = xgb_preds, val_fe['RUL_clipped']

plt.figure(figsize=(6, 6))
plt.scatter(y_to_plot, preds_to_plot, alpha=0.4, s=15)
lims = [0, max(max(y_to_plot), max(preds_to_plot))]
plt.plot(lims, lims, 'r--', label='Perfect prediction')
plt.xlabel('Actual RUL'); plt.ylabel('Predicted RUL')
plt.title(f'Actual vs Predicted RUL — {best_overall}')
plt.legend(); plt.tight_layout(); plt.show()

## Next
Phases 4–7 are complete. **09_survival_analysis.ipynb** (Phase 9), **10_model_comparison.ipynb**, and **11_shap_analysis.ipynb** (Phase 10) remain placeholders — Phase 8 (Failure Risk Modeling) reuses the same feature datasets built here with a classification target derived from RUL thresholds, and will get its own notebook when implemented.